Cài đặt các thư viện cần thiết

In [11]:
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

from IPython.display import clear_output

workspace_root = Path.cwd()
melo_dir = workspace_root / "MeloTTS"
openvoice_dir = workspace_root / "OpenVoice"
checkpoint_zip = workspace_root / "checkpoints_v2_0417.zip"
checkpoint_url = "https://myshell-public-repo-host.s3.amazonaws.com/openvoice/checkpoints_v2_0417.zip"


def replace_double_equals(file_path: Path) -> None:
    original_text = file_path.read_text(encoding="utf-8")
    updated_text = original_text.replace("==", ">=")
    if updated_text != original_text:
        file_path.write_text(updated_text, encoding="utf-8")


subprocess.run([sys.executable, "-m", "pip", "install", "pydub"], check=True)

if not melo_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/myshell-ai/MeloTTS.git", str(melo_dir)], check=True)

replace_double_equals(melo_dir / "setup.py")
replace_double_equals(melo_dir / "requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(melo_dir)], check=True)
subprocess.run([sys.executable, "-m", "unidic", "download"], check=True)

if not openvoice_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/myshell-ai/OpenVoice.git", str(openvoice_dir)], check=True)

replace_double_equals(openvoice_dir / "requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(openvoice_dir / "requirements.txt")], check=True)

urllib.request.urlretrieve(checkpoint_url, checkpoint_zip)
with zipfile.ZipFile(checkpoint_zip) as archive:
    archive.extractall(workspace_root)

if checkpoint_zip.exists():
    checkpoint_zip.unlink()

clear_output()
print("Đã cài đặt thư viện, chỉnh sửa requirements và giải nén checkpoint xong.")

Đã cài đặt thư viện, chỉnh sửa requirements và giải nén checkpoint xong.


Biến môi trường thiết Iập có in chi tiết không?

In [12]:
DETAIL_PRINT = False

Import các thư viện cần thiết:

In [13]:
import joblib
import json
import os
import copy
import sys
import numpy as np
import torch
from pathlib import Path
import pdb
import inspect
import shutil
from datetime import datetime
import gdown
from zoneinfo import ZoneInfo
import cv2
from scipy.spatial.distance import cdist
from scipy.spatial.distance import euclidean

# Lấy thời gian hiện tại với múi giờ Hồ Chí Minh
now = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh"))
# Định dạng thành xâu YYMMDD_hhmm
AT_TIME = now.strftime("%y%m%d_%H%M")
CAM_MAP = {
    "test" : "fs_"
}
TYPE_SET = "test"

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
# Đặt font mặc định thành sans-serif
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Liberation Sans', 'Bitstream Vera Sans']

Cần thêm chức năng tách được mp4 (không có âm thanh) và mp3 (giọng đọc Ấn) ra khỏi video gốc:

In [14]:
VIDEO_IDS = {
    "Variable Characteristics.mp4" : "2Q0TpVYet3A" #link youtube: https://www.youtube.com/watch?v=2Q0TpVYet3A
}

import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("imageio_ffmpeg") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "imageio-ffmpeg"], check=True)
import imageio_ffmpeg
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

if importlib.util.find_spec("yt_dlp") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "yt-dlp"], check=True)

import yt_dlp

video_id = VIDEO_IDS["Variable Characteristics.mp4"]
video_url = f"https://www.youtube.com/watch?v={video_id}"


def download_video_only(url: str, output_name: str) -> None:
    output_path = Path(output_name)
    if output_path.exists():
        print(f"{output_name} đã tồn tại, bỏ qua tải xuống video.")
        return

    ydl_opts = {
        "format": "bestvideo[ext=mp4][vcodec^=avc1]/bestvideo[ext=mp4]/bestvideo",
        "outtmpl": str(output_path.with_suffix(".%(ext)s")),
        "noplaylist": True,
        "quiet": False,
        "ffmpeg_location": ffmpeg_exe,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        downloaded_path = Path(ydl.prepare_filename(info))

    if downloaded_path != output_path:
        if downloaded_path.suffix.lower() != ".mp4":
            subprocess.run([
                ffmpeg_exe,
                "-y",
                "-i",
                str(downloaded_path),
                "-an",
                "-c:v",
                "copy",
                str(output_path),
            ], check=True)
            if downloaded_path.exists():
                downloaded_path.unlink()
        else:
            downloaded_path.replace(output_path)


def download_audio_only(url: str, output_name: str) -> None:
    output_path = Path(output_name)
    if output_path.exists():
        print(f"{output_name} đã tồn tại, bỏ qua tải xuống audio.")
        return

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": str(output_path.with_suffix(".%(ext)s")),
        "noplaylist": True,
        "quiet": False,
        "ffmpeg_location": ffmpeg_exe,
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            }
        ],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        downloaded_path = Path(ydl.prepare_filename(info))

    mp3_path = downloaded_path.with_suffix(".mp3")
    if mp3_path.exists() and mp3_path != output_path:
        mp3_path.replace(output_path)
    elif downloaded_path.exists() and downloaded_path.suffix.lower() == ".mp3" and downloaded_path != output_path:
        downloaded_path.replace(output_path)


download_video_only(video_url, "video_no_audio.mp4")
download_audio_only(video_url, "Variable Characteristics.mp3")

video_no_audio.mp4 đã tồn tại, bỏ qua tải xuống video.
Variable Characteristics.mp3 đã tồn tại, bỏ qua tải xuống audio.


Code xử lý Chunking và chuyển đổi giọng

In [15]:
GOOGLE_DRIVE_IDS = {
    #Dùng id này nếu không tách được mp3 khỏi mp4 "Variable Characteristics.mp3" : "10UCA_EYi_Bw06fDYpSXzOOk1sejhCn-X",
    "sample_American_accent.mp3" : "1wwJZXG7GnrJ8j_IPwCwvbUtpU3_vPkJ9"#file ID của tệp giọng đọc chuẩn Anh Mỹ
}

Hàm tải tệp về Notebook

In [16]:
def download_files():
    """Tải tệp từ Google Drive nếu chưa tồn tại cục bộ"""
    for filename, file_id in GOOGLE_DRIVE_IDS.items():
        if not os.path.exists(filename):
            if DETAIL_PRINT:
              print(f"Đang tải {filename} từ Google Drive...")
            url = f'https://drive.google.com/uc?id={file_id}'
            gdown.download(url, filename, quiet=False)
        else:
            if DETAIL_PRINT:
              print(f"Tệp {filename} đã tồn tại, bỏ qua tải xuống.")
download_files()


Chuẩn bị chuyển đổi

In [ ]:
import os
import sys
import subprocess
import torch

# 1. Cài đặt static-ffmpeg vào môi trường ảo nếu chưa có
try:
    import static_ffmpeg
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "static-ffmpeg"], check=True)
    import static_ffmpeg

# 2. Bơm cả ffmpeg và ffprobe vào PATH cục bộ của phiên chạy hiện tại
static_ffmpeg.add_paths()

# 3. Chỉ đường cho Python tìm thấy module openvoice
sys.path.append(os.path.abspath("OpenVoice"))

from openvoice import se_extractor
from openvoice.api import ToneColorConverter
from pydub import AudioSegment

# Khởi tạo mô hình OpenVoice (Tự động nhận diện GPU)
ckpt_converter = 'checkpoints_v2/converter'
device = "cuda:0" if torch.cuda.is_available() else "cpu"
tone_color_converter = ToneColorConverter(f'{ckpt_converter}/config.json', device=device)
tone_color_converter.load_ckpt(f'{ckpt_converter}/checkpoint.pth')

def split_audio(file_path, chunk_length_ms=30000, min_chunk_ms=10000):
    """Cắt audio thành các đoạn nhỏ 30 giây. Nếu đoạn cuối quá ngắn, gộp vào đoạn liền trước."""
    audio = AudioSegment.from_file(file_path)
    chunks = []
    
    # Cắt các đoạn 30s
    for i in range(0, len(audio), chunk_length_ms):
        chunks.append(audio[i:i + chunk_length_ms])
    
    # Kiểm tra và gộp đoạn cuối nếu quá ngắn
    if len(chunks) > 1 and len(chunks[-1]) < min_chunk_ms:
        last_chunk = chunks.pop()
        chunks[-1] += last_chunk
        
    chunk_paths = []
    os.makedirs("temp_chunks", exist_ok=True)
    for i, chunk in enumerate(chunks):
        chunk_name = f"temp_chunks/chunk_{i}.wav"
        chunk.export(chunk_name, format="wav")
        chunk_paths.append(chunk_name)
    return chunk_paths

def convert_accent(input_file, target_reference, output_name="final_american_voice.wav"):
    print("1. Đang trích xuất đặc trưng giọng chuẩn (Anh-Mỹ)...")
    # Lấy "màu giọng" (Tone color) từ file mẫu
    target_se, _ = se_extractor.get_se(target_reference, tone_color_converter, target_dir='processed', vad=True)

    print("2. Đang cắt nhỏ file 6 phút...")
    chunk_paths = split_audio(input_file)
    processed_chunks = []

    print(f"3. Bắt đầu xử lý {len(chunk_paths)} đoạn...")
    for idx, chunk_path in enumerate(chunk_paths):
        print(f"   -> Đang xử lý đoạn {idx + 1}/{len(chunk_paths)}")
        out_chunk_path = f"temp_chunks/processed_chunk_{idx}.wav"

        # Trích xuất đặc trưng của đoạn âm thanh gốc
        source_se, _ = se_extractor.get_se(chunk_path, tone_color_converter, target_dir='processed', vad=True)

        # Áp dụng Voice Conversion (giữ nguyên nhịp điệu, đổi accent)
        tone_color_converter.convert(
            audio_src_path=chunk_path,
            src_se=source_se,
            tgt_se=target_se,
            output_path=out_chunk_path,
            message="@MyShell"
        )

        # Lưu trữ lại đoạn đã xử lý
        processed_chunks.append(AudioSegment.from_wav(out_chunk_path))

    print("4. Đang nối các đoạn lại thành file hoàn chỉnh...")
    final_audio = AudioSegment.empty()
    for chunk in processed_chunks:
        final_audio += chunk

    final_audio.export(output_name, format="wav")
    print(f"Hoàn tất! File kết quả được lưu tại: {output_name}")

# ==========================================
# CẤU HÌNH FILE ĐẦU VÀO CỦA BẠN TẠI ĐÂY
# ==========================================
input_audio = "Variable Characteristics.mp3"       # Thay bằng tên file thực tế tải lên Colab
reference_audio = "sample_American_accent.mp3"          # File mẫu giọng Mỹ (10-15s)
output_audio = "American_accent_Variable_Characteristics.wav"           # Tên file xuất ra

# Gọi hàm thực thi
convert_accent(input_audio, reference_audio, output_audio)


Download of https://github.com/zackees/ffmpeg_bins/raw/main/v8.0/win32.zip -> c:\Users\dequyce\Downloads\Py\.venv\lib\site-packages\static_ffmpeg\bin\win32.zip completed.
Extracting c:\Users\dequyce\Downloads\Py\.venv\lib\site-packages\static_ffmpeg\bin\win32.zip -> c:\Users\dequyce\Downloads\Py\.venv\lib\site-packages\static_ffmpeg\bin


Loaded checkpoint 'checkpoints_v2/converter/checkpoint.pth'
missing/unexpected keys: [] []
1. Đang trích xuất đặc trưng giọng chuẩn (Anh-Mỹ)...
OpenVoice version: v2
[(0.0, 10.482), (11.278, 13.362), (17.038, 28.818), (29.326, 32.146), (33.39, 42.61), (44.782, 50.5249375)]
after vad: dur = 42.12800453514739


c:\Users\dequyce\Downloads\Py\.venv\lib\site-packages\torch\functional.py:681: UserWarning: stft with return_complex=False is deprecated. In a future pytorch release, stft will return complex tensors for all inputs, and return_complex=False will raise an error.
Note: you can still call torch.view_as_real on the complex output to recover the old return format. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\SpectralOps.cpp:879.)
  return _VF.stft(  # type: ignore[attr-defined]


2. Đang cắt nhỏ file 6 phút...
3. Bắt đầu xử lý 14 đoạn...
   -> Đang xử lý đoạn 1/14
OpenVoice version: v2
[(0.43, 30.0)]
after vad: dur = 29.57
   -> Đang xử lý đoạn 2/14
OpenVoice version: v2
[(0.0, 1.362), (1.55, 18.13), (18.766, 30.0)]
after vad: dur = 29.176
   -> Đang xử lý đoạn 3/14
OpenVoice version: v2
[(0.11, 17.202), (17.646, 30.0)]
after vad: dur = 29.446
   -> Đang xử lý đoạn 4/14
OpenVoice version: v2
[(0.0, 5.298), (5.358, 30.0)]
after vad: dur = 29.94
   -> Đang xử lý đoạn 5/14
OpenVoice version: v2
[(0.0, 7.314), (7.63, 30.0)]
after vad: dur = 29.684
   -> Đang xử lý đoạn 6/14
OpenVoice version: v2
[(0.0, 14.034), (14.446, 30.0)]
after vad: dur = 29.588
   -> Đang xử lý đoạn 7/14
OpenVoice version: v2
[(0.0, 30.0)]
after vad: dur = 30.0
   -> Đang xử lý đoạn 8/14
OpenVoice version: v2
[(0.0, 5.138), (5.326, 21.618), (21.774, 30.0)]
after vad: dur = 29.656
   -> Đang xử lý đoạn 9/14
OpenVoice version: v2
[(0.0, 19.986), (20.11, 24.274), (24.59, 30.0)]
after vad: dur = 

AssertionError: input audio is too short

In [ ]:
import subprocess
from pathlib import Path
import imageio_ffmpeg

ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

input_video = Path("video_no_audio.mp4")
input_audio = Path("American_accent_Variable_Characteristics.wav")
output_video = Path("Final_Variable_Characteristics_American.mp4")

subprocess.run(
    [
        ffmpeg_exe,
        "-y",
        "-i",
        str(input_video),
        "-i",
        str(input_audio),
        "-c:v",
        "copy",
        "-c:a",
        "aac",
        str(output_video),
    ],
    check=True,
)

print(f"Đã tạo {output_video}")